<a href="https://colab.research.google.com/github/frank2720/TF-IDF/blob/main/TF_IDF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import requests
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import re


TARGET_TERM = "seo services"

urls = {
    "your_page": "https://www.pudfradigital.com/our-services/search-engine-optimization",
    "competitor_1": "https://kenseo.co.ke/",
    "competitor_2": "https://artlydigitalmarketing.co.ke/top-seo-agency-in-kenya/",
    "competitor_3": "https://www.seosmart.co.ke/",
}


def fetch_clean_text(url):
    headers = {"User-Agent": "Mozilla/5.0"}
    html = requests.get(url, headers=headers, timeout=15).text
    soup = BeautifulSoup(html, "lxml")

    for tag in soup(["script", "style", "nav", "footer", "header", "noscript"]):
        tag.decompose()

    text = soup.get_text(separator=" ")
    text = re.sub(r"\s+", " ", text)
    return text.lower()

# BUILD CORPUS
documents = {}
for name, url in urls.items():
    documents[name] = fetch_clean_text(url)
    print(f"Fetched: {name}")

# TF-IDF
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 4)
)

tfidf_matrix = vectorizer.fit_transform(documents.values())

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    index=documents.keys(),
    columns=vectorizer.get_feature_names_out()
)

# OUTPUT TF-IDF SCORE
term = TARGET_TERM.lower()

print("\nTF-IDF scores for:", TARGET_TERM, "\n")
for doc in tfidf_df.index:
    score = tfidf_df.loc[doc, term] if term in tfidf_df.columns else 0
    print(f"{doc}: {score:.6f}")


Fetched: your_page
Fetched: competitor_1
Fetched: competitor_2
Fetched: competitor_3

TF-IDF scores for: seo services 

your_page: 0.043438
competitor_1: 0.068898
competitor_2: 0.054927
competitor_3: 0.011058
